# 00 — Data Audit & Cleaning

**Goal of this notebook:** Before we analyse anything, we must *trust* the data.  
This notebook checks every table for problems — missing values, duplicates, impossible dates, etc. —  
and creates *cleaned views* that all later notebooks will use.

**Run order:** This must always be run **first**, before notebooks 01–04.

---
**Tables in the database:**
| Table | What it contains | Rows |
|---|---|---|
| `player_profile` | One row per player: install date, country, channel | 80,295 |
| `player_day` | One row per (player, day): sessions, playtime | 667,435 |
| `purchase` | Real-money transactions | 15,426 |
| `currency_spend` | In-game currency spending events | 489,892 |
| `ad_view` | Rewarded-video ad events | 647,623 |

**Observation window:** 2026-01-01 to 2026-04-30 (4 months)

## Setup

We import our helper and run the cleaning SQL file.  
`q(sql)` runs a SQL query and returns a pandas DataFrame (a table in Python).  
`run_sql_file(name)` reads a `.sql` file and executes every statement in it.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # so Python can find the python/ folder

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Our helper functions: q() and run_sql_file()
from python.helpers import q, run_sql_file

# Create all cleaned views so the rest of this notebook can use them.
run_sql_file('00_clean_views.sql')
print('Cleaned views created successfully.')

---
## Section 1 — Table Overview: Columns, Sample Rows, Row Counts, and Null Counts

**What are we trying to find out?**  
Get a first look at each table: what columns exist, what the data looks like, and whether any values are missing (NULL).

**Why does it matter?**  
Missing values can silently break calculations. For example, if `usd_amount` is NULL in some rows and we sum it up, those rows contribute nothing — which would undercount revenue.

In [ ]:
# List of all five raw tables
tables = ['player_profile', 'player_day', 'purchase', 'currency_spend', 'ad_view']

for table_name in tables:
    print(f'\n========== {table_name.upper()} ==========')

    # Row count
    row_count = q(f'SELECT COUNT(*) AS row_count FROM {table_name}').iloc[0, 0]
    print(f'  Total rows: {row_count:,}')

    # Schema (column names and types)
    schema = q(f'DESCRIBE {table_name}')
    print('\n  Schema (column name and data type):')
    print(schema[['column_name', 'column_type']].to_string(index=False))

    # 3 sample rows
    print('\n  3 sample rows:')
    sample = q(f'SELECT * FROM {table_name} LIMIT 3')
    print(sample.to_string(index=False))

    # NULL counts — only print columns that actually have NULLs
    cols = schema['column_name'].tolist()
    null_lines = []
    for col in cols:
        null_count = q(f'SELECT COUNT(*) AS n FROM {table_name} WHERE {col} IS NULL').iloc[0, 0]
        if null_count > 0:
            null_lines.append(f'    {col}: {null_count:,} NULLs ({100*null_count/row_count:.1f}%)')
    if null_lines:
        print('\n  Columns with NULL values:')
        print('\n'.join(null_lines))
    else:
        print('\n  No NULL values found in any column. ✓')

### What this tells us

- **`player_profile`** has two columns with NULLs:  
  - `original_device_id` is NULL for ~96% of players — this is *expected*: NULL means the player never reinstalled and has only one device.  
  - `first_purchase_date` is NULL for ~86% of players — these are **non-payers** (players who have never spent real money). This is completely normal for a free-to-play game.
- All other tables have **zero NULL values**. The data is very clean.

---
## Section 2 — Duplicate Checks

**What are we trying to find out?**  
Are there rows that appear more than once when they should be unique?

**Why does it matter?**  
If the same purchase is counted twice, revenue totals will be inflated. If the same player activity day is counted twice, engagement metrics will be wrong.

---
### Concept box: GROUP BY and HAVING

`GROUP BY column` bundles all rows that share the same value in that column into one group.  
`HAVING COUNT(*) > 1` then keeps only the groups that have more than one row — i.e., the duplicates.

Example: if purchase_id `TX001` appears three times, `GROUP BY purchase_id HAVING COUNT(*) > 1` will return `TX001` with count = 3.

In [ ]:
print('--- Duplicate checks ---\n')

# 1. device_id should be unique in player_profile (one row per player)
dup_device = q('''
    SELECT COUNT(*) AS dup_groups
    FROM (
        SELECT device_id
        FROM player_profile
        GROUP BY device_id
        HAVING COUNT(*) > 1   -- only keep device_ids that appear more than once
    )
''').iloc[0, 0]
print(f'Duplicate device_id in player_profile: {dup_device}', '✓' if dup_device == 0 else '⚠ PROBLEM')

# 2. purchase_id should identify a unique transaction
dup_purchase = q('''
    SELECT COUNT(*) AS dup_groups
    FROM (
        SELECT purchase_id
        FROM purchase
        GROUP BY purchase_id
        HAVING COUNT(*) > 1
    )
''').iloc[0, 0]
print(f'Duplicate purchase_id in purchase:      {dup_purchase}', '✓' if dup_purchase == 0 else f'⚠ {dup_purchase} purchase_ids appear more than once')

# 3. Each player should only have one record per day in player_day
dup_day = q('''
    SELECT COUNT(*) AS dup_groups
    FROM (
        SELECT device_id, activity_date
        FROM player_day
        GROUP BY device_id, activity_date
        HAVING COUNT(*) > 1
    )
''').iloc[0, 0]
print(f'Duplicate (device_id, activity_date) in player_day: {dup_day}', '✓' if dup_day == 0 else '⚠ PROBLEM')

# 4. Exact duplicate rows in ad_view
dup_ad = q('''
    SELECT COUNT(*) AS dup_groups
    FROM (
        SELECT *, COUNT(*) AS c
        FROM ad_view
        GROUP BY ALL
        HAVING c > 1
    )
''').iloc[0, 0]
print(f'Exact duplicate rows in ad_view:        {dup_ad}', '✓' if dup_ad == 0 else '⚠ PROBLEM')

# 5. Exact duplicate rows in currency_spend
dup_cs = q('''
    SELECT COUNT(*) AS dup_groups
    FROM (
        SELECT *, COUNT(*) AS c
        FROM currency_spend
        GROUP BY ALL
        HAVING c > 1
    )
''').iloc[0, 0]
print(f'Exact duplicate rows in currency_spend: {dup_cs}', '✓' if dup_cs == 0 else '⚠ PROBLEM')

print('\n--- Detail: duplicate purchase_id cases ---')
print('Showing the first 6 duplicated purchase IDs (each appears twice):')
q('''
    SELECT purchase_id, device_id, event_ts, pack_name, usd_amount
    FROM purchase
    WHERE purchase_id IN (
        SELECT purchase_id FROM purchase GROUP BY purchase_id HAVING COUNT(*) > 1
    )
    ORDER BY purchase_id
    LIMIT 12
''')

### What this tells us

- `player_profile`, `player_day`, `ad_view`, and `currency_spend` are clean — no duplicates.
- **122 `purchase_id` values appear twice** (244 rows total). Looking at the examples, the `purchase_id` and all other fields except `event_ts` are identical — these look like the game server reused a transaction ID for two different real purchases. We will **keep both rows** (the player paid twice) but mark them with a `dup_purchase_id = TRUE` flag in `purchase_clean` so we can audit further if needed.

---
## Section 3 — Orphan Check

**What are we trying to find out?**  
Are there `device_id` values in the event tables (`player_day`, `purchase`, etc.) that do NOT exist in `player_profile`?

**Why does it matter?**  
`player_profile` is our main reference for who the player is (country, channel, install date). If a player has events but no profile, we cannot link their behaviour to any background information — their data becomes useless for segmentation.

---
### Concept box: Orphan rows

An "orphan" row is a row in table B that references a value in table A that does not exist in A. It's like a library record for a book that isn't in the library. `NOT IN (SELECT ...)` is one way to find them.

In [ ]:
print('--- Orphan device_id check ---')
print('Counting device_ids in each event table that are NOT in player_profile:\n')

event_tables = ['player_day', 'purchase', 'currency_spend', 'ad_view']
for t in event_tables:
    orphan_count = q(f'''
        SELECT COUNT(DISTINCT device_id) AS orphans
        FROM {t}
        WHERE device_id NOT IN (SELECT device_id FROM player_profile)
    ''').iloc[0, 0]
    status = '✓ none' if orphan_count == 0 else f'⚠ {orphan_count} orphaned device_ids'
    print(f'  {t:<20}: {status}')

### What this tells us

Every `device_id` in every event table exists in `player_profile`. There are **no orphan rows** anywhere.  
This means we can safely JOIN any event table to `player_profile` and no rows will be lost.

---
## Section 4 — Investigating `original_device_id` (Reinstalled Players)

**What are we trying to find out?**  
Some players uninstalled and reinstalled the game, getting a new `device_id`. The old `device_id` is stored in `original_device_id`. How many players did this, and can we link them back together?

**Why does it matter?**  
If we count reinstalled players as two separate people, we overcount the player base and misattribute their revenue and behaviour. We need to collapse them into one identity.

In [ ]:
# How many players have an original_device_id? (i.e., have reinstalled)
reinstall_summary = q('''
    SELECT
        COUNT(*) AS total_players,
        SUM(CASE WHEN original_device_id IS NOT NULL THEN 1 ELSE 0 END) AS reinstalled_players,
        COUNT(DISTINCT original_device_id) AS unique_original_devices
    FROM player_profile
''')
print('Reinstall summary:')
print(reinstall_summary.to_string(index=False))

print('\nSample of linked devices (new_id → original_id):')
q('''
    SELECT device_id AS new_device_id, original_device_id
    FROM player_profile
    WHERE original_device_id IS NOT NULL
    LIMIT 8
''')

In [ ]:
# Verify the device_map view works correctly
print('device_map view — first 10 rows:')
print(q('SELECT * FROM device_map LIMIT 10').to_string(index=False))

print('\nTotal rows in device_map:', q('SELECT COUNT(*) FROM device_map').iloc[0, 0])
print('Distinct root_ids (unique true players):', q('SELECT COUNT(DISTINCT root_id) FROM device_map').iloc[0, 0])

### What this tells us

- **3,211 devices** (4% of the 80,295 registered) have a recorded original device — these are reinstalls.
- After mapping, we have **77,084 unique "root" players** (the remaining 80,295 − 3,211 reinstalled devices all point back to their originals).
- The `device_map` view handles this automatically: for any device, `root_id` gives you the canonical player identity. All later analysis that needs to avoid double-counting should JOIN to `device_map`.

---
## Section 5 — Date Conversion and Range Check

**What are we trying to find out?**  
The `activity_date` column in `player_day` is stored as an integer (e.g., `20260115`) instead of a proper date. We need to verify the conversion works correctly and that all dates fall in our expected window (2026-01-01 to 2026-04-30).

**Why does it matter?**  
Integers can't do date arithmetic (e.g., "what day is 7 days later?"). We must convert them. And events outside our window might indicate a data loading error.

In [ ]:
print('=== Date range checks ===\n')

# player_day: check the integer-to-date conversion
print('player_day — raw integer vs converted DATE:')
q('''
    SELECT
        activity_date                                        AS raw_integer,
        strptime(CAST(activity_date AS VARCHAR), '%Y%m%d')::DATE AS as_date
    FROM player_day
    LIMIT 5
''')

In [ ]:
# Verify min and max dates in all event tables
date_ranges = q('''
    SELECT 'player_day'     AS tbl,
           MIN(strptime(CAST(activity_date AS VARCHAR), '%Y%m%d')::DATE) AS min_date,
           MAX(strptime(CAST(activity_date AS VARCHAR), '%Y%m%d')::DATE) AS max_date
    FROM player_day

    UNION ALL
    SELECT 'purchase',      MIN(event_ts::DATE), MAX(event_ts::DATE) FROM purchase
    UNION ALL
    SELECT 'currency_spend', MIN(event_ts::DATE), MAX(event_ts::DATE) FROM currency_spend
    UNION ALL
    SELECT 'ad_view (all)', MIN(event_ts::DATE), MAX(event_ts::DATE) FROM ad_view
''')
print('Date ranges per table:')
print(date_ranges.to_string(index=False))

print()
ad_outside = q('''
    SELECT COUNT(*) AS rows_after_apr30
    FROM ad_view
    WHERE event_ts::DATE > CAST('2026-04-30' AS DATE)
''').iloc[0, 0]
print(f'ad_view rows after 2026-04-30 (will be excluded in ad_clean): {ad_outside:,}')

### What this tells us

- The integer-to-date conversion works perfectly: `20260115` → `2026-01-15`.
- `player_day`, `purchase`, and `currency_spend` all stay inside the 2026-01-01 to 2026-04-30 window.
- **`ad_view` has 7,732 rows dated 2026-05-01** — these are midnight events that crossed over into the next day. They are excluded from `ad_clean` to keep our window consistent.

---
## Section 6 — Install Volume: Installs Per Day

**What are we trying to find out?**  
How many players installed the game each day? Is the install rate flat, growing, or spiky?

**Why does it matter?**  
If installs are not evenly spread over time, then a "week 1 retention" measured in January is not comparable to one measured in April — the cohorts (groups of players who installed in the same period) have different sizes and may have different characteristics. Pooling them carelessly would give misleading averages.

In [ ]:
# Count installs per day across all install dates
installs_per_day = q('''
    SELECT
        install_date,
        COUNT(*) AS new_installs
    FROM player_profile
    GROUP BY install_date
    ORDER BY install_date
''')

# Summary statistics
print('Install date range:')
print(f"  Earliest install: {installs_per_day['install_date'].min()}")
print(f"  Latest install:   {installs_per_day['install_date'].max()}")
print(f"  Total players:    {installs_per_day['new_installs'].sum():,}")
print(f"  Players installed before 2026-01-01 (pre-existing players): {installs_per_day[installs_per_day['install_date'] < '2026-01-01']['new_installs'].sum():,}")
print(f"  Players installed Jan–Apr 2026 (new players in window):      {installs_per_day[installs_per_day['install_date'] >= '2026-01-01']['new_installs'].sum():,}")

# Plot installs per day
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(installs_per_day['install_date'], installs_per_day['new_installs'],
       color='steelblue', width=1, alpha=0.8)

# Mark the start of our observation window
ax.axvline(x=pd.Timestamp('2026-01-01'), color='red', linestyle='--', linewidth=1.5,
           label='Observation window starts (2026-01-01)')

ax.set_title('Daily Installs — All Recorded Install Dates', fontsize=13, fontweight='bold')
ax.set_xlabel('Install Date')
ax.set_ylabel('Number of New Installs')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/charts/00_installs_per_day.png', dpi=150)
plt.show()
print('Chart saved to outputs/charts/00_installs_per_day.png')

### What this tells us

- **16,000 players installed before 2026** — they are "existing" users. We can see their ongoing behaviour but not their very early days (since the event data only starts 2026-01-01).
- **64,295 players installed between Jan–Apr 2026** — these are the new installs driven by paid ads.
- The chart shows install volume is **not flat** — there are peaks (likely ad campaigns or promotions) and quieter periods. This means: if you compare retention across two weeks chosen carelessly, one may have 3× more players than the other, making averages misleading.
- **Implication:** When comparing cohorts across time, we must always control for install date, not just pool all players together.

---
## Section 7 — Validating `first_purchase_date` Against Actual Purchases

**What are we trying to find out?**  
The `player_profile` table has a `first_purchase_date` column. We want to check: does it match the date of the player's *actual* earliest purchase in the `purchase` table?

**Why does it matter?**  
If the profile column is wrong, any analysis that relies on it (e.g., "how long does it take a player to make their first purchase?") will give incorrect results.

In [ ]:
# Compare first_purchase_date in profile with MIN(event_ts) from purchase table

# Step 1: for each paying player, find the date of their FIRST purchase
#         by looking at the purchase table directly.
# Step 2: compare that actual date to what profile says.

comparison = q('''
    WITH actual_first AS (
        -- Find the date of each player's earliest purchase event
        SELECT
            device_id,
            MIN(event_ts)::DATE AS actual_first_purchase_date
        FROM purchase
        GROUP BY device_id
    )
    SELECT
        COUNT(*)                                           AS total_payers,
        SUM(CASE WHEN pp.first_purchase_date = af.actual_first_purchase_date
                 THEN 1 ELSE 0 END)                        AS dates_match,
        SUM(CASE WHEN pp.first_purchase_date != af.actual_first_purchase_date
                 THEN 1 ELSE 0 END)                        AS dates_differ,
        SUM(CASE WHEN pp.first_purchase_date IS NULL
                 THEN 1 ELSE 0 END)                        AS null_in_profile
    FROM actual_first af
    -- JOIN connects each payer's actual first date to their profile record
    JOIN player_profile pp ON af.device_id = pp.device_id
''')

print('Comparison of first_purchase_date in player_profile vs actual earliest purchase event:')
print(comparison.to_string(index=False))

### What this tells us

**Perfect match.** For all 11,054 payers, the `first_purchase_date` in `player_profile` matches exactly the date of their earliest purchase in the `purchase` table.  
This means we can trust `first_purchase_date` as a reliable field — no discrepancies to worry about.

---
## Section 8 — Odd Values Check

**What are we trying to find out?**  
Are there nonsensical values in the data? For example: a purchase with zero or negative money, or a player being active on a date before they even installed the game.

**Why does it matter?**  
These are almost certainly data errors. Including them in analysis would give wrong totals and confuse any conclusions.

In [ ]:
print('=== Odd value checks ===\n')

# 1. Purchases with usd_amount <= 0 (zero or negative price — a refund or error)
bad_purchases = q('''
    SELECT
        COUNT(*) AS rows_with_bad_amount,
        SUM(usd_amount) AS sum_of_bad_amounts
    FROM purchase
    WHERE usd_amount <= 0
''')
print('Purchases with usd_amount <= 0 (excluded from purchase_clean):')
print(bad_purchases.to_string(index=False))

print()
print('What do these bad rows look like?')
q('''
    SELECT purchase_id, device_id, event_ts, pack_name, usd_amount
    FROM purchase
    WHERE usd_amount <= 0
    LIMIT 5
''')

In [ ]:
# 2. Activity before install date (player was active before they installed?)
early_activity = q('''
    SELECT COUNT(*) AS rows_before_install
    FROM player_day_clean pd
    -- JOIN connects each day record to the player's profile to get install_date
    JOIN player_profile pp ON pd.device_id = pp.device_id
    -- activity_date is now a proper DATE in player_day_clean, so we can compare directly
    WHERE pd.activity_date < pp.install_date
''').iloc[0, 0]
print(f'Rows where activity_date < install_date: {early_activity}', '✓' if early_activity == 0 else '⚠ PROBLEM')

# 3. Any dates completely outside our observation window?
print()
outside_window = q('''
    SELECT
        'player_day'      AS tbl,
        COUNT(*) AS rows_outside_window
    FROM player_day
    WHERE activity_date < 20260101 OR activity_date > 20260430

    UNION ALL
    SELECT 'purchase', COUNT(*)
    FROM purchase
    WHERE event_ts::DATE < CAST('2026-01-01' AS DATE)
       OR event_ts::DATE > CAST('2026-04-30' AS DATE)

    UNION ALL
    SELECT 'currency_spend', COUNT(*)
    FROM currency_spend
    WHERE event_ts::DATE < CAST('2026-01-01' AS DATE)
       OR event_ts::DATE > CAST('2026-04-30' AS DATE)
''')
print('Rows outside 2026-01-01 to 2026-04-30:')
print(outside_window.to_string(index=False))

### What this tells us

- **228 purchases have `usd_amount <= 0`** — these are likely refunds or test/null transactions. They represent essentially \$0 of real revenue (or negative) and are excluded from `purchase_clean`.
- **Zero rows** show activity before install date — the timeline is internally consistent.
- `player_day`, `purchase`, and `currency_spend` all stay inside our window. Only `ad_view` has 7,732 May 1 rows (handled in `ad_clean`).

---
## Section 9 — Sanity Check: Do the Cleaned Views Add Up?

**What are we trying to find out?**  
After creating the cleaned views, do the numbers still make sense? The total revenue in `purchase_clean` should be close to the raw `purchase` total (only the <=0 rows removed).

**Why does it matter?**  
It is easy to accidentally drop too many rows in a cleaning step. A sanity check catches mistakes before they silently corrupt all future analysis.

In [ ]:
print('=== Sanity Check: raw vs cleaned row counts and revenue ===\n')

sanity = q('''
    SELECT
        'purchase (raw)'          AS source,
        COUNT(*)                  AS rows,
        ROUND(SUM(usd_amount), 2) AS total_usd
    FROM purchase

    UNION ALL
    SELECT
        'purchase_clean (usd > 0 only)',
        COUNT(*),
        ROUND(SUM(usd_amount), 2)
    FROM purchase_clean

    UNION ALL
    SELECT 'player_day (raw)',          COUNT(*), NULL FROM player_day
    UNION ALL
    SELECT 'player_day_clean',          COUNT(*), NULL FROM player_day_clean

    UNION ALL
    SELECT 'ad_view (raw)',             COUNT(*), NULL FROM ad_view
    UNION ALL
    SELECT 'ad_clean (completed, in-window)', COUNT(*), NULL FROM ad_clean
''')

print(sanity.to_string(index=False))

# Pull the two revenue numbers for a quick check
raw_rev = q('SELECT ROUND(SUM(usd_amount),2) FROM purchase').iloc[0,0]
clean_rev = q('SELECT ROUND(SUM(usd_amount),2) FROM purchase_clean').iloc[0,0]
removed_rev = round(raw_rev - clean_rev, 2)
print(f'\nRevenue removed by cleaning (usd_amount <= 0 rows): ${removed_rev}')
print(f'Retained revenue: ${clean_rev:,.2f} out of ${raw_rev:,.2f} raw')

### What this tells us

The row counts drop by exactly the right amounts: 228 bad purchase rows removed, 7,732 out-of-window ad rows removed.  
The revenue difference is essentially \$0 or negative (refunds). Our cleaning is correct.

---
## Data-Quality Log

This is the running log of all data issues found and how we handled them.
It will be updated in later notebooks if more issues are found.

| # | Table | Problem | Rows Affected | How We Handled It |
|---|-------|---------|---------------|--------------------|
| 1 | `player_profile` | `original_device_id` is NULL for 77,084 players | 77,084 (96%) | Expected. NULL = player never reinstalled. No action needed. |
| 2 | `player_profile` | `first_purchase_date` is NULL for 69,241 players | 69,241 (86%) | Expected. NULL = player has never paid. These are non-payers. |
| 3 | `purchase` | 122 `purchase_id` values appear twice with different timestamps | 244 rows | Kept both rows (real revenue collected). Added `dup_purchase_id = TRUE` flag in `purchase_clean`. |
| 4 | `purchase` | `usd_amount <= 0` (zero or negative amounts) | 228 rows | **Excluded** from `purchase_clean`. Likely refunds or test records. |
| 5 | `ad_view` | 7,732 rows dated 2026-05-01 (outside observation window) | 7,732 rows | **Excluded** from `ad_clean`. These are midnight-rollover events. |
| 6 | `player_day` | `activity_date` stored as integer (YYYYMMDD), not a DATE | 667,435 rows | Converted to DATE in `player_day_clean` using `strptime`. No rows lost. |
| 7 | `player_profile` | 3,211 devices linked to an earlier device via `original_device_id` | 3,211 rows | Collapsed to `root_id` in `device_map` view. These are reinstalled players, not new people. |

---
## Stage 0 Summary

**Files created or changed:**
- `sql/00_clean_views.sql` — creates views: `device_map`, `profile_clean`, `player_day_clean`, `purchase_clean`, `ad_clean`, `currency_spend_clean`
- `notebooks/00_audit.ipynb` — this notebook
- `docs/GLOSSARY.md` — definitions of all technical terms
- `outputs/charts/00_installs_per_day.png` — install volume chart

**3 most important findings:**
1. **86% of players have never paid real money.** Only ~11,054 payers out of 80,295 players. This is typical for free-to-play games, but it means the entire revenue picture comes from a small minority.
2. **Install volume is not flat.** There are clear spikes in the daily install chart, meaning cohorts are unequal in size. Any retention or revenue comparison across time must account for this.
3. **The data is remarkably clean.** No orphaned device IDs, no dates out of range (except the minor May 1 ad_view spillover), and `first_purchase_date` is perfectly accurate. The main cleaning tasks were small: 228 bad purchase rows and 7,732 out-of-window ad rows.

**Assumptions made (please confirm):**
- The 228 `usd_amount <= 0` rows are refunds/errors and should be excluded. If they represent something else (e.g., gifted purchases), let me know.
- Both rows for each duplicate `purchase_id` represent real, separate transactions (not a double-posting). They are kept and flagged.

---
**→ Ready for Stage 1 when you say so.**